# Part G — Diagonalization

**Final Project: The Linear Algebra Behind AI — A Regression Problem**

**Research Title: Predicting Software Execution Time Using Code Complexity Metrics**

Once a matrix is diagonalized, keeping only the most important directions becomes a simple, principled way to reduce dimensionality.

---

**Student Names:**

1. Mary Yvonne Carandang
2. Kelsey Leigh Cebrero
3. John Elvin Endrenal
4. Mark Joseph Holgado
5. Rolan Kian Sacopon

**Date:** November 20, 2026

---

**Chosen Dataset:** BigO(Bench)

**Dataset Source:** Hugging Face

**Dataset Link:** https://huggingface.co/datasets/facebook/BigOBench

**What to do:** Diagonalize your matrix from Part F (C = VΛV⁻¹, or C = VΛVᵀ if symmetric) and use it to reduce dimensionality.

In [ ]:
# Load the transformed data from Part D and eigenpairs from Part F.
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
data_d = np.load('/content/drive/MyDrive/part_d_data.npz')
data_f = np.load('/content/drive/MyDrive/part_f_data.npz')
X_train = data_d['X_train']
X_test = data_d['X_test']
y_train = data_d['y_train']
y_test = data_d['y_test']
C = data_f['C']
eigvals = data_f['eigvals']
eigvecs = data_f['eigvecs']

In [ ]:
# Import numerical and plotting libraries used for diagonalization and projection.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [ ]:
# Assemble the eigenvector matrix V and diagonal eigenvalue matrix Lambda.
V = eigvecs
Lambda = np.diag(eigvals)

# Verify V^T V = I (orthogonality) and that V Lambda V^T reconstructs C
reconstructed_C = V @ Lambda @ V.T
print("V^T V = I ?", None if V is None else np.allclose(V.T @ V, np.eye(V.shape[0])))
print("Reconstruction matches C ?", None if reconstructed_C is None else np.allclose(C, reconstructed_C))

In [ ]:
# Verify orthogonality and reconstruct the covariance matrix from its eigendecomposition.
explained_var_ratio = eigvals / eigvals.sum()
cum_var = np.cumsum(explained_var_ratio)
k = int(np.searchsorted(cum_var, 0.95) + 1)
print(f"k = {k}, cumulative variance explained = {None if k is None else cum_var[k-1]*100:.2f}%")

In [ ]:
# Select the smallest number of principal components explaining at least 95 percent of variance.
V_k = V[:, :k]
Z_train = X_train @ V_k
Z_test = X_test @ V_k
print("Z_train shape:", None if Z_train is None else Z_train.shape)

In [ ]:
# Project training and test features into the retained eigenvector subspace.
plt.scatter(Z_train[:, 0], Z_train[:, 1], c=y_train, cmap="viridis")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.colorbar(label="target y")
plt.title("Data projected onto the eigenbasis")
plt.show()

### Answer these in your report

**Q25.** Write your diagonalization in the form C = VΛV⁻¹ (or VΛVᵀ). What are the dimensions of V and Λ?

> *Your answer:* Since $C$ is symmetric, its diagonalization is $C = V\Lambda V^T$, where `V = eigvecs` and `Lambda = diag(eigvals)`. With one feature, `V` and `Lambda` are both $1 \times 1$ matrices.

**Q26.** How did you verify, numerically, that your diagonalization is correct?

> *Your answer:* I checked that `V.T @ V` equals the identity matrix and reconstructed the covariance matrix using `V @ Lambda @ V.T`. The notebook prints both checks; both should be `True` within floating-point tolerance.

**Q27.** How many components did you keep to explain at least 95% of the variance? Why did you choose that threshold?

> *Your answer:* I selected `k` as the smallest index whose cumulative explained variance is at least 0.95. For this one-feature dataset, `k = 1`, explaining 100% of the variance. The 95% threshold preserves nearly all information while still allowing dimensionality reduction when more features are present.

**Q28.** What is lost — mathematically and practically — when you reduce dimensionality this way? Give a concrete, specific answer for your data.

> *Your answer:* Mathematically, discarded eigenvector directions and their variance are lost, so the original data cannot generally be reconstructed exactly. In this dataset, no information is lost because the only feature is retained (`k = 1`); with additional low-variance features, their variation would be discarded and predictions depending on them could change.

### Save your work for the next notebook

In [ ]:
# Save the retained eigenvectors and projected data for Part H.
from google.colab import drive
drive.mount('/content/drive')

np.savez('/content/drive/MyDrive/part_g_data.npz', V_k=V_k, Z_train=Z_train, Z_test=Z_test)
print("Part G saved successfully!")